# 겉보기 무관 회귀(SUR)로 주거용 에너지 수요 시스템 추정하기

## 요약

한 지역 유틸리티가 **PROC SYSLIN**의 겉보기 무관 회귀(seemingly-unrelated-regression, SUR) 방법을 사용해 두 방정식으로 이루어진 주거용 **에너지 수요 시스템** — 자기 가격, 교차 가격, 가구 소득의 함수로서 전기와 천연가스 예산 비중 — 을 추정합니다. 두 비중 방정식은 공통의 가구 예산을 공유하므로 오차가 교차 상관됩니다. SUR은 그 상관을 활용하도록 방정식을 공동으로 추정하여, 부호가 일관된 자기 및 교차 가격 효과를 복원하고, 수요 분석가에게 필요한 방정식 간 공분산과 제약 검정을 제공합니다.

## 데이터 소스

| 데이터셋 | 행 수 | 단위 | 주요 변수 | 설명 |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | 월별 시장 관측치당 한 행 | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | 합성 월별 주거용 에너지 시장 패널. `lp_elec`/`lp_gas`는 전기와 천연가스의 로그 실질 가격, `lincome`은 로그 실질 가구 소득, `w_elec`/`w_gas`는 전기와 천연가스의 지출 예산 비중으로, 알려진 AIDS 스타일 수요 구조에 상관된 방정식 간 잡음을 더해 생성했다. |

# 겉보기 무관 회귀로 주거용 에너지 수요 시스템 추정하기

한 지역 가스·전기 유틸리티는 상대 가격과 소득이 바뀔 때 가구가 **전기**와 **천연가스** 사이에서 에너지 예산을 어떻게 재배분하는지 이해하고자 합니다. 자연스러운 틀은 **수요 시스템**입니다: 공동으로 추정되는 예산 비중 방정식들의 집합.

두 가지 특징이 공동 추정을 올바른 도구로 만듭니다:

- 비중 방정식들은 공통의 가구 예산에 기반하므로 오차가 **교차 상관**됩니다 — 가구가 전기에 더 쓰면 가스에는 덜 씁니다. 방정식을 **겉보기 무관 회귀(SUR)**로 함께 추정하면 그 상관을 효율성에 활용합니다.
- 경제 이론은 시스템 추정량이 직접 강제할 수 있는 **선형 제약**(합산, 동차성, 대칭성)을 부과합니다.

`SUR` 옵션을 붙인 `PROC SYSLIN`은 바로 이 상황을 위해 만들어졌습니다. 각 비중 방정식을 적합하고, 방정식 간 오차 공분산을 추정한 뒤, 그 공분산으로 시스템을 다시 적합하여 효율적이고 이론에 일관된 탄력성을 제공합니다 — 방정식 간 공분산 행렬과 공동 제약 검정과 함께.

이 노트북에서 우리는 다음을 수행합니다:

1. 알려진 비중 구조로부터 합성 월별 에너지 시장 패널을 생성합니다.
2. 두 방정식 비중 시스템을 SUR로 적합합니다.
3. 공동 SUR 적합을 방정식별 OLS와 비교합니다.
4. 동차성 제약을 부과하고 그 공동 F-검정을 읽습니다.
5. 탄력성 보고를 위해 계수 표를 추출합니다.

## 1단계 — 합성 에너지 시장 패널 생성

우리는 작은 **두 재화 에너지 수요 시스템**(전기와 천연가스)의 월별 관측치 60개를 시뮬레이션합니다. 데이터 생성 과정은 선형화된 AIDS 스타일 예산 비중 모델입니다:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

우리는 **방정식 간 상관**을 의도적으로 심습니다: 가구가 전기에 더 쓰면 가스에 덜 쓰므로 `e1`과 `e2`는 음의 상관을 가집니다. 공통의 에너지 시장 가격 요인 또한 두 로그 가격을 함께 움직입니다. 이것들이 각 방정식을 따로 적합하는 것보다 공동 SUR 추정에 보상을 주는 특징입니다.

In [1]:
데이터 energy;
   호출 streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   반복 month = 1 까지 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      출력;
   종료;

   유지 month lp_elec lp_gas lincome w_elec w_gas;
실행;

처리 평균 데이터=energy n mean std MIN MAX maxdec=3;
   변수 w_elec w_gas lp_elec lp_gas lincome;
실행;

                                                  The MEANS Procedure

 Variable        N           Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------
 W_ELEC         60          0.440       0.011       0.417       0.464
 W_GAS          60          0.228       0.014       0.198       0.256
 LP_ELEC        60          4.617       0.142       4.354       4.911
 LP_GAS         60          4.211       0.162       3.818       4.566
 LINCOME        60         10.916       0.088      10.723      11.174
 --------------------------------------------------------------------



NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 2단계 — 수요 시스템의 기준 SUR 추정

우리는 두 비중 방정식을 공동으로 추정합니다. `SUR` 옵션은 `PROC SYSLIN`에게 방정식 간 오차 공분산을 추정하고 이를 효율적인 실현 가능 GLS(feasible-GLS) 적합에 사용하도록 지시합니다. 두 개의 라벨이 붙은 `MODEL` 문(`elec`과 `gas`)이 시스템을 정의하며, 각각은 예산 비중을 두 로그 가격과 로그 소득에 회귀시킵니다.

SYSLIN은 각 방정식의 모수 추정치를 보고하고, 마지막에 **방정식 간 공분산 행렬(Cross-Model Covariance Matrix)** — SUR이 활용하는 두 방정식 오차의 추정 공분산 — 을 보고합니다.

In [2]:
처리 syslin 데이터=energy ON;
   elec: 모형 w_elec = lp_elec lp_gas lincome;
   gas:  모형 w_gas  = lp_elec lp_gas lincome;
실행;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## 3단계 — 방정식별 OLS와 비교

SUR이 우리에게 무엇을 주는지 보기 위해, 같은 두 방정식을 보통 최소제곱법(기본 방법, 한 번에 한 방정식)으로 다시 적합합니다. OLS는 방정식 간 오차 상관을 무시하므로, 그 상관이 0이 아닐 때 — 여기서는 구성상 그렇습니다 — 일치추정량이지만 SUR보다 덜 효율적입니다.

두 계수 표를 비교하면 시스템 구조가 반영될 때 추정치와 그 표준오차가 어떻게 이동하는지 드러납니다.

In [3]:
처리 syslin 데이터=energy ols;
   elec: 모형 w_elec = lp_elec lp_gas lincome;
   gas:  모형 w_gas  = lp_elec lp_gas lincome;
실행;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## 4단계 — 경제 이론 부과와 검정

수요 이론은 명목 가격/소득 효과가 **0차 동차성(homogeneity of degree zero)**을 따라야 한다고 말합니다: 모든 가격과 소득을 같은 배율로 조정해도 실질 수요는 변하지 않으므로, 한 방정식 내 가격과 소득 계수는 합이 0이어야 합니다. 우리는 이를 `RESTRICT` 문으로 전기 비중에 부과합니다.

SYSLIN은 제약을 받는 시스템을 다시 적합하고, 그 제약이 데이터와 일관되는지에 대한 **제약 결과(Restriction Results)** F-검정 — 동차성 가설의 직접적인 공동 검정 — 을 보고합니다.

In [4]:
처리 syslin 데이터=energy ON;
   elec: 모형 w_elec = lp_elec lp_gas lincome;
   gas:  모형 w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
실행;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## 5단계 — 탄력성 보고를 위한 추정치 포착

최종 보고서를 위해 우리는 수렴된 계수를 `OUTEST=`로 보존합니다. 결과 데이터셋은 방정식당 한 행에 절편과 기울기 추정치, 그리고 적합 통계량을 담으며, 이는 표본 평균 비중에서의 유틸리티 자기 및 교차 가격 탄력성 계산에 투입됩니다. 기록을 위해 이를 출력합니다.

In [5]:
처리 syslin 데이터=energy ON outest=demand_est;
   elec: 모형 w_elec = lp_elec lp_gas lincome;
   gas:  모형 w_gas  = lp_elec lp_gas lincome;
실행;

처리 인쇄 데이터=demand_est noobs;
   제목 "SUR demand-system coefficient estimates";
실행;
제목;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## 결과 해석

**부호 일관성.** SUR 추정치는 데이터에 구축된 대체 구조를 복원합니다. 전기 비중 방정식에서 **자기 로그 가격 계수는 양수**(`LP_ELEC` = 0.112)이고 **교차 가격 계수는 음수**(`LP_GAS` = -0.098)이며, 가스 비중 방정식에서는 그 패턴이 대칭입니다(자기 `LP_GAS` = 0.114, 교차 `LP_ELEC` = -0.075). 모든 자기 및 교차 가격 효과가 통계적으로 유의하므로(각 `Pr > |t|`가 0.005 미만), 대체 부호는 잡음 섞인 적합의 인공물이 아니라 확고하게 식별됩니다.

**SUR은 방정식 간 상관을 활용한다.** **방정식 간 공분산 행렬**은 음의 비대각 원소(-0.000068)를 보여 줍니다: 가구는 전기 지출과 가스 지출을 맞바꾸며, 구성한 그대로입니다. 그 공분산이 0이 아니므로 공동 SUR 추정은 각 방정식을 따로 적합하는 것보다 효율적입니다 — 2단계의 SUR 표준오차는 3단계의 방정식별 OLS 표준오차보다 한결같이 약간 더 작으며(예를 들어 가스 `LP_ELEC` 표준오차는 OLS 하의 0.0264에서 SUR 하의 0.0255로 떨어짐), 점추정치는 변하지 않습니다. 그 더 팽팽한 추론이 두 개의 별개 회귀가 아니라 시스템을 모델링한 보상입니다.

**이론이 성립한다.** `RESTRICT`를 통해 전기 비중에 **0차 동차성**(가격과 소득 계수의 합이 0)을 부과하니 제약 결과 F-검정은 2.14, `Pr > F` = 0.149가 나왔습니다. 제약은 **기각되지 않으므로**, 소비자 수요 이론의 동차성 예측은 이 표본과 일관됩니다 — 유틸리티는 제약된, 이론에 일관된 추정치를 자신 있게 보고에 가져갈 수 있습니다.

**운영 활용.** `OUTEST=` 표는 방정식당 한 행에 절편과 기울기 추정치, 적합 통계량을 보존합니다. 그 계수들은 표본 평균 비중(1단계에서 `W_ELEC` ~ 0.44, `W_GAS` ~ 0.23)에서 자기 및 교차 가격 탄력성과 소득(지출) 탄력성으로 곧바로 변환되어, 유틸리티에 수요 예측과 요율 설계 시나리오를 위한 방어 가능하고 이론에 일관된 근거를 제공합니다.